# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farahhussain159-create/flyrank-ml-week1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Which content pages are most likely already declining in search performance, and how should a content team prioritize refresh effort across a limited review capacity?

Decision it supports: Content strategists at FlyRank-style clients need to decide, each cycle, which of thousands of pages to review or refresh first. A wrong call costs either wasted refresh effort (false positive) or a missed declining page that keeps losing traffic (false negative).

Lane: Refresh / Content Opportunity Scoring — reusing and validating a Random Forest classifier (from Week 5) against a hand-written baseline rule (Week 4), with an honest client-grouped validation split (Week 6) and a final ranked action playbook (Week 7).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Dataset: FlyRank ML Internship dataset ("FlyRank/internship-warehouse" on Hugging Face), Pseudonymized Warehouse Release v20260703. The full warehouse contains approximately 81.8 million rows (daily fact table: 78,835,655 rows) across a star schema with salted, namespaced, fingerprinted hash keys.

Tables used: For this capstone, two slices were used — (1) a 30,000-row anonymized content-performance sample (content_refresh_anonymized.csv, 44 columns) for baseline scoring and signal auditing, and (2) a 9,841,378-row daily performance fact table slice (fact_content_daily_performance, month=2026-03) for the data contract and leakage audit.

Date window: March 2026 (2026-03-01 to 2026-03-31), 31 unique days, used as the mid-panel month for the data contract.

What was excluded and why (public-safe): The fact_content_daily_performance_sample.parquet file (the sealed June 2026 test month) was deliberately excluded from all feature-building and label-tuning steps, to preserve it as an unbiased final holdout. Raw client identifiers, domains, and search queries were never accessed in identifiable form — all analysis uses anonymized/hashed IDs (client_hash_id, content_hash_id), per the FlyRank Internship Data Use Terms (anonymized research and education use only; no re-identification attempts; no redistribution of raw data; no client-identifying data in any public output).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions: Declining search performance can be approximated by trend_direction == "down" (label: is_declining_label), a proxy derived from observed traffic/ranking trend rather than a direct causal measure of "needs refresh."

Features used: Same-day, decision-time signals only — impressions_90d, avg_position, content_age_days, char_count, word_count, days_since_last_update, search_volume, ctr, plus GSC/GA4 fields (gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions) in the data-contract slice. All features are pre-computed 90-day windows or static attributes — none derived from trend_direction or trend_pct.

Label definition: is_declining_label = 1 if trend_direction == "down", else 0. Class balance: 54.21% declining in the 30k sample.

Baseline: A hand-written rule combining a staleness score (from freshness_tier) and a normalized search-volume score (60%/40% weighted average), producing an action_score and a ranked queue. Baseline Precision@50 = 0.38 (naive random split, Week 4/5 comparison).

Model: Random Forest Classifier (n_estimators=300, max_depth=8) and Logistic Regression, trained on the same feature set and label.

Validation design: GroupShuffleSplit by client_id/content_hash_id (80/20), ensuring no client's rows appear in both train and test — the honest, deployment-realistic split.

Leakage checks: A dedicated leakage audit (Week 3 + Week 6) tested: (1) naive random split vs. grouped split — Precision@50 dropped from 0.840 (naive, client leakage possible) to 0.580 (honest, grouped); (2) deliberately injecting the true label as a feature — R² jumped from an honest 0.311 to a leaky 0.377 (mild leak) and to a fake 1.0 (label smuggled in directly), confirming the test harness correctly detects leakage. All final features passed a timeline check (no feature derived from the label window), label-derived/sibling check, and product-flag check — all clean.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Model vs. baseline, same honest split:

Method	Precision@50	Precision (0.5 cutoff)	Recall (0.5 cutoff)	F1 (0.5 cutoff)
Baseline (rule)	0.38	—	—	—
Logistic Regression	0.60	0.546	0.573	0.559
Random Forest	0.52	0.565	0.742	0.642

Both trained models beat the hand-written baseline rule on Precision@50 under the same random-row split used in Week 4/5. However, re-running the same Logistic Regression model under an honest, client-grouped split (Week 6) showed Precision@50 drop from 0.840 (naive split, client leakage possible) to 0.580 (honest split) — a large gap that means part of the apparent skill was memorization of client-specific patterns, not genuine predictive signal.

Errors (Random Forest, honest test set, n=6,163): 1,796 false positives (flagged as declining but weren't) and 812 false negatives (missed a real decline). Top drivers of the model's predictions: impressions_90d, avg_position, content_age_days.

Ranked action queue output (Week 7 playbook): Of 6,163 honestly-scored test items — P1 (review this week): 160 items; P2 (review this month): 2,906 items; P3 (low priority/monitor): 3,097 items.

[Charts to embed here: priority-tier bar chart and feature-importance chart, from work/figures/w07_priority_tiers_and_features.png]

## 5. Limitations

*What this work cannot claim.*

This work has real limits that any reader should weigh before acting on it.

Scale and generalization: Trained and tested on one FlyRank-style anonymized dataset (30,000 rows, ~32 clients for the modeling slice). It has not been validated on other content types, industries, or a different data time window (all analysis here uses the March 2026 slice).

What Precision@50 means: A Precision@50 of 0.58 (honest, grouped split) means roughly 4 in 10 items in a top-50 pull are false alarms. The queue narrows human attention — it does not replace human judgment on any single item.

Observed error rates: 1,796 false positives and 812 false negatives were observed on the held-out test set — these are measured, not hypothetical, error rates.

Correlation, not causation: This is cross-sectional/observational data with no controlled experiment. The model finds association between features (impressions, position, content age) and decline labels — it cannot support a causal claim like "refreshing X will improve Y."

No time-series validation: The honest split confirms the model doesn't memorize specific clients, but it has not been re-tested on data from a later time window to confirm performance holds as content and behavior drift over time (see Section 6 monitoring triggers for why this matters).

Not tested for fairness/bias: No fairness or bias audit was run across client segments or content categories — a real deployment would need one before use across an unreviewed population.

Label is a proxy: is_declining_label is derived from an observed trend flag, not a verified ground-truth outcome — it's the best available proxy, not a perfect one.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Based on the Random Forest model's honest, held-out scoring (6,163 items), here is the ranked action playbook for a content team:

1. P1 — Review this week (160 items, highest priority). These carry the highest decline_probability scores and the strongest combined reason codes (stale content 90+ days since update, low search ranking, below-median impressions, old content 180+ days). Action: human reviewer opens each flagged item and confirms it's not a false positive (e.g., a recently updated page not yet reflected in the 90-day window) before taking any refresh action.

2. P2 — Review this month (2,906 items, moderate priority). Solid signal but lower confidence than P1. Action: batch review on a monthly cadence rather than urgent action.

3. P3 — Low priority / monitor only (3,097 items). Weak or mixed signal — often low search volume or a single weak driver. Action: no immediate refresh; monitor for signal changes over the next cycle.

4. Human review is mandatory before any action — the model's output is a triage aid, not an auto-action trigger. A second reviewer should sign off before P1 content is unpublished, merged, or majorly rewritten.

5. Never use this system to: auto-publish or auto-delete content without a human in the loop; make client-facing causal claims ("your content is declining because of X"); rank or score individual employees based on how much of their content gets flagged; apply it to a client or content type outside the training distribution without re-validation first.

6. Monitor for model decay: Re-run Precision@50 on a fresh labeled sample periodically (monthly). A drop meaningfully below the validated 0.580 baseline, or a sudden jump in flag-rate (e.g., from ~3% to 15%+), signals either a real content-base shift or a broken feature pipeline — investigate before trusting the queue. Retrain (not just re-score) when the 90-day feature window has rolled forward significantly, when new clients/content types are added, or when more than ~3–6 months have passed since the last training run.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Artifacts generated by this pipeline, ready for the deployed paper:

Ranked action queue (CSV): work/outputs/w07_ranked_action_queue.csv — 6,163 rows, columns: rank, content_id, client_id, decline_probability, priority_tier, reason_code, days_since_last_update, avg_position, impressions_90d, content_age_days, actual_declining.
Metrics receipts (JSON): work/outputs/w07_playbook_metrics.json — committed metrics the paper's numbers trace back to: model type, split method, precision_at_50_grouped (0.58), precision/recall/F1 at 0.5 cutoff, false positive/negative counts, test set size, priority tier counts, top features.
Chart 1 — Priority tier distribution: bar chart of P1/P2/P3 item counts (work/figures/w07_priority_tiers_and_features.png, left panel).
Chart 2 — Feature importance: horizontal bar chart of the top 8 features driving decline_probability from the trained Random Forest (work/figures/w07_priority_tiers_and_features.png, right panel).
Baseline action score CSV: work/outputs/baseline_action_score.csv — the Week 4 rule-based ranked queue, used for baseline comparison.

These artifacts are the traceable source for every number reported in the Results and Recommendations sections of the paper.

## 8. 5-Minute Demo Outline (Week-8 Showcase — optional)

**Question (30s):** Out of a large content catalog, which pages should a team refresh first when it can't review everything?

**Method (1 min):** A Random Forest classifier trained on FlyRank's anonymized warehouse data, validated with a client-grouped split (GroupShuffleSplit) to avoid leakage, benchmarked against FlyRank's existing hand-written rule-based flag.

**One chart (1.5 min):** `work/figures/w07_priority_tiers_and_features.png` — left: how the ranked queue splits into P1/P2/P3 review tiers; right: the top features (impressions_90d, avg_position, content_age_days) driving the score.

**One honest result (1 min):** On the honest, client-grouped split, the model scored Precision@50 = 0.58 vs. the existing rule's 0.38 — a real but modest lift, with 1,796 false positives and 812 false negatives out of 6,163 held-out items.

**One recommendation (1 min):** Use the three-tier queue (P1/P2/P3) as a triage aid for human reviewers, never as an auto-action system. Re-check Precision@50 monthly — a drop signals drift and a need to retrain.

## 9. Shareable Cuts

**Social post (methodology):**
Beat FlyRank's rule-based content-decay flag with a Random Forest model — Precision@50 of 0.58 vs. 0.38, measured on an honest client-grouped split (no data leakage). Trained on real anonymized search-performance data. Sharing the chart + the honest limitations, not just the win. #MachineLearning #SEO

**Employer-facing summary (3 sentences):**
I built a Random Forest classifier that flags declining content pages for refresh prioritization, trained and validated on a client-grouped, leakage-checked split of FlyRank's anonymized search-performance warehouse data (~9.8M rows sampled). It outperformed FlyRank's existing hand-written rule-based flag at Precision@50 (0.58 vs. 0.38) while producing a transparent three-tier action queue with clearly reported error rates (1,796 false positives, 812 false negatives on 6,163 held-out items). The system is designed as a human-in-the-loop triage aid, not an automated decision-maker.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
